# EDA - Heart Disease (UCI id=45, Cleveland)
Análise exploratória univariada e bivariada. Base do app Streamlit (`app.py`).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

try:
    df = pd.read_csv("../data/heart_disease.csv")
except FileNotFoundError:
    df = pd.read_csv("data/heart_disease.csv")
df.head()

## 1. Inspeção estrutural

In [ ]:
print(df.shape)
print("Duplicatas:", df.duplicated().sum())
df.dtypes

## 2. Binarização da variável alvo
`num` = 0 → Normal; 1 a 4 → Cardíaco.

In [ ]:
df['doenca_cardiaca'] = df['num'].apply(lambda x: 'Normal' if x == 0 else 'Cardiaco')
print(df['doenca_cardiaca'].value_counts())

## 3. Dados ausentes

In [ ]:
faltantes = df.isna().sum()
pd.DataFrame({'faltantes': faltantes, '%': (faltantes / len(df) * 100).round(2)})[faltantes > 0]

In [ ]:
# Ausências ocorrem em registros distintos?
print(df[['ca', 'thal']].isna().value_counts())

Ausências < 2% da amostra (`ca` e `thal`). Estratégia: remoção simples (`dropna`), sem viés relevante.

In [ ]:
df = df.dropna(subset=['ca', 'thal']).copy()
print(df.shape)

## 4. Análise univariada

In [ ]:
quant = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
df[quant].describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 7))
for i, c in enumerate(quant):
    axes[0, i].hist(df[c], bins=25)
    axes[0, i].set_title(f'Histograma: {c}')
    axes[1, i].boxplot(df[c])
    axes[1, i].set_title(f'Boxplot: {c}')
plt.tight_layout()
plt.show()

In [ ]:
for c in ['sex', 'cp', 'exang']:
    print(df[c].value_counts(), '\n')

## 5. Análise bivariada (vs. diagnóstico)

In [ ]:
df.groupby('doenca_cardiaca')[quant].mean().round(2)

In [ ]:
df.boxplot(column='thalach', by='doenca_cardiaca')
plt.title('Frequência cardíaca máxima por diagnóstico')
plt.suptitle('')
plt.show()

In [ ]:
pd.crosstab(df['cp'], df['doenca_cardiaca'], normalize='index').round(2)

In [ ]:
df['alvo'] = (df['num'] > 0).astype(int)
corr = df[quant + ['alvo']].corr()
plt.figure(figsize=(7, 5))
plt.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar()
plt.xticks(range(len(corr)), corr.columns, rotation=45)
plt.yticks(range(len(corr)), corr.columns)
plt.title('Matriz de correlação')
plt.show()
corr['alvo'].sort_values()

## 6. Conclusão
- Dataset limpo (303 → 297 linhas), alvo binarizado e relações bivariadas consolidadas.
- Pacientes com doença atingem menor frequência cardíaca máxima (`thalach`) e maior `oldpeak`.
- Dor assintomática (`cp` = 4) concentra a maior proporção de diagnósticos positivos.

*Testes de hipótese (Qui-quadrado, Mann-Whitney) omitidos por serem opcionais no escopo exploratório.*